In [5]:
import requests
import time
import random
import os
import bs4
import htmlmin
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


name = "Konosuba"
ID = "tt5370118"
SEASON = 3
HAVE_SPECIAL = False

name = "Overlord"
ID = "tt4869896"
SEASON = 4
HAVE_SPECIAL = False

name = "Mairimashita! Iruma-kun"
ID = "tt11034066"
SEASON = 3
HAVE_SPECIAL = False

name = "Boku no Hero Academia"
ID = "tt5626028"
SEASON = 6
HAVE_SPECIAL = True

name = "Tensei Shitara Slime Datta Ken"
ID = "tt9054364"
SEASON = 2
HAVE_SPECIAL = True

main_path = os.path.join("raw", name)
os.makedirs(main_path, exist_ok=True)

firefox_options = Options()
firefox_options.add_argument("--headless")
service = Service('./geckodriver')
driver = webdriver.Firefox(service=service)

def download_season(url, number):
    driver.get(url)
    # Encontrar y hacer clic en el botón con clase "ipc-see-more__button" hasta que ya no exista
    """
    while True:
        try:
            # Esperar hasta que el botón esté disponible y hacer clic en él
            button = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.CLASS_NAME, "ipc-see-more__button"))
            )
            button.click()
        except:
            # Si no se encuentra el botón, romper el bucle
            break
    """
    time.sleep(5)
    
    # Guardar el HTML resultante
    html = driver.page_source
    
    # Escribir el HTML en un archivo
    path = os.path.join(main_path, number)
    with open(path, 'w', encoding='utf-8') as file:
        file.write(html)

for x in range(1, SEASON+1):
    download_season(f"https://www.imdb.com/title/{ID}/episodes/?season={x}", f"T{x}.html")
if HAVE_SPECIAL:
    download_season(f"https://www.imdb.com/title/{ID}/episodes/?season=Unknown", f"TX.html")

driver.quit()

In [6]:
import requests
import time
import random
import os
import bs4
import htmlmin
from datetime import datetime, date
import locale

def join(texto):
    return " ".join([x for x in texto.split(" ") if len(x) > 0])

files = sorted(filter(lambda x: x.endswith(".html"), os.listdir(f"raw/{name}")))

locale.setlocale(locale.LC_TIME, 'es_ES.UTF-8')
meses = ['ene', 'feb', 'mar', 'abr', 'may', 'jun', 
         'jul', 'ago', 'sept', 'oct', 'nov', 'dic',
]

with open(f"data/{name}.tsv", encoding="UTF-8", mode="w") as file:
    header = "id, imdb_season, chapter, score, votes, release, canon, title, subtitle, episode_imdb_url, img_imdb_url"
    file.write("\t".join(header.split(", ")) + "\n")
    id_ = 0
    for x in files:
        name_f = f"raw/{name}/{x}"
        print(name_f)
        with open(name_f) as html:
            soup = bs4.BeautifulSoup(html.read())
            for element in soup.findAll("article", {"class": "episode-item-wrapper"}):
                img = element.find("img").get("src")
                url = "https://m.imdb.com" + element.find("a", {"class": "ipc-lockup-overlay"}).get("href").split("?ref")[0]
                title = join(element.find("div", {"class": "ipc-title__text"}).text.replace("\n", ""))
                try:
                    subtitle = join(element.find("div", {"class": "ipc-html-content-inner-div"}).text.replace("\n", ""))
                except AttributeError:
                    subtitle = "Unknown"
                fecha = join(element.find("span", {"class": "sc-ccd6e31b-10"}).text.replace("\n", ""))
                votes = element.find("span", {"class": "ipc-rating-star--voteCount"}).text.replace("\n", "").strip().strip("()")
                score = element.find("span", {"class": "ipc-rating-star--base"}).text.replace("\n", "")
                score = score.split("/")[0].replace(",", ".")
    
                if " ∙ " in title:
                    number, title = title.split(" ∙ ")
                    try:
                        season, chapter = list(map(lambda x: x[1:], number.split(".")))
                    except:
                        print(number)
                else:
                    season = "Unknown"
                    chapter = "Unknown"

                dia, mes, año = fecha.split(", ")[1].split(" ")
                mes = meses.index(mes) + 1
                fecha = date(int(año), mes, int(dia)).strftime("%Y-%m-%d")
    
                data = [str(id_+1), season, chapter, score, votes, fecha, "canon", title, subtitle, img, url]
                
                print("\t".join(data), file=file)
                id_+= 1

raw/Tensei Shitara Slime Datta Ken/T1.html
raw/Tensei Shitara Slime Datta Ken/T2.html
raw/Tensei Shitara Slime Datta Ken/TX.html


In [7]:
import requests
import time
import os


service = Service('./geckodriver')
driver = webdriver.Firefox(service=service)
os.makedirs(f"img/{name}", exist_ok=True)
with open(f"data/{name}.tsv", encoding="UTF-8") as file:
    file.readline()
    for line in file:
        id_, season, chapter, score, votes, fecha, canon, title, subtitle, img, url = line.strip().split("\t")
            
        if os.path.exists(f"img/{name}/img_{id_}.jpg"):
            continue

        nombre_archivo = f"img/{name}/img_{id_}.jpg"
        driver.get(img)
        with open(nombre_archivo, 'wb') as archivo:
            archivo.write(driver.find_element(By.TAG_NAME, 'img').screenshot_as_png)

        time.sleep(2)
        
driver.quit()

In [8]:
import json

with open("data.json", "r", encoding="UTF-8") as file:
    data = json.load(file)

data.append({"name": name})

with open("data.json", "w", encoding="UTF-8") as file:
    json.dump(data, file, indent=4)